In [3]:
import os 
import gc 
import cv2 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max

In [50]:
image = "images/20260202_pdau4.jpg"
#image = "images/20260202_pdau1.jpg"
img = cv2.imread(image)
image = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_, binary = cv2.threshold(image, 200, 255, cv2.THRESH_BINARY)
kernel = np.ones((10,10),np.uint8)
closing = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
contour, _ = cv2.findContours(closing, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

In [ ]:
points = []
scalebar_pixel = 0
for i, cnt  in enumerate(contours):
  area = cv2.contourArea(cnt)
  x, y, w, h = cv2.boundingRect(cnt)
  
  if area > 200:
    x, y, w, h = cv2.boundingRect(cnt)
    yn, ym = y - 2, y + h + 2
    xn, xm = x - 2, x + w + 2
    roi = image[yn:ym, xn:xm]
    _, bin = cv2.threshold(roi, 200, 255, cv2.THRESH_BINARY)
    roi_edge = cv2.Canny(bin, threshold1=50, threshold2=150)
    lines = cv2.HoughLinesP(roi_edge, 1, np.pi/180, threshold=200, minLineLength=200, maxLineGap=1)
    if lines is not None:
      pixel_row = np.sum(bin, axis = 1)
      y_pos = np.argmax(pixel_row)
      column = np.where(bin[y_pos, :] == 255)[0]
      xe = column[-1]
      xs = column[0]
      scalebar_pixel = xe - xs + 1
  points.append([[x - 2, y - 2], 
      [x + w + 2, y - 2], 
      [x + w + 2, y + h + 2], 
      [x - 2, y + h + 2]])
scalebar_length = 0 #input("Measurement of Scalebar(no units): ")

def Conversion(pixels):
  measurement = (pixels * scalebar_length) / scalebar_pixel
  return measurement 

In [ ]:
shapes_array = np.array(points)
img_h, img_w = img.shape[:2]
global_mask1 = np.zeros((img_h, img_w), dtype=np.uint8)
global_mask2 = np.zeros((img_h, img_w), dtype=np.uint8)
dt = 1049
for i, cnt  in enumerate(contours):
  x, y, w, h = cv2.boundingRect(cnt)

  if i == dt:
    xn, xm = x - 2, x + w + 2
    yn, ym = y - 2, y + h + 2
    bounds1 = [yn, xn, ym, xm]
    roi = image[yn:ym, xn:xm]
    _, bin = cv2.threshold(roi, 200, 255, cv2.THRESH_BINARY)
    nearby_corners_indexes = []
    for target_point in points[dt]:
      search_radius = 50
      corner_distances = np.linalg.norm(shapes_array - target_point, axis=2)
      inside_mask = corner_distances <= search_radius
      matching_indices = np.any(inside_mask, axis=1)
      original_indices = np.flatnonzero(matching_indices)
      inside_shapes = shapes_array[original_indices]
      for nearby in original_indices:
        nearby_corners_indexes.append(nearby)
    nearby_corners_indexes = np.unique(nearby_corners_indexes)
    nearby_corners_indexes = nearby_corners_indexes[nearby_corners_indexes != dt]
    for nearby_contours in nearby_corners_indexes:
      x_val, xm_val = points[nearby_contours][0][0], points[nearby_contours][2][0]
      y_val, ym_val = points[nearby_contours][0][1], points[nearby_contours][2][1]
      roi_two = image[y_val:ym_val, x_val:xm_val]
      _, binr = cv2.threshold(roi_two, 200, 255, cv2.THRESH_BINARY)
      bounds2 = [y_val, x_val, ym_val, xm_val]
      global_mask1[yn:ym, xn:xm] = bin[0:(ym-yn), 0:(xm-xn)]
      global_mask2[y_val:ym_val, x_val:xm_val] = binr[0:(ym_val-y_val), 0:(xm_val-x_val)]

    # 4. Perform bitwise AND on the identical-sized global masks
      overlapping_white_mask = cv2.bitwise_and(global_mask1, global_mask2)

    # 5. Extract coordinates of overlapping white pixels inside the local overlap region
    # np.where returns (y_indices, x_indices)
      white_pixel_y, white_pixel_x = np.where(overlapping_white_mask == 255)

      if len(white_pixel_x) == 0:
        continue

    # 6. Translate the local white pixel boundaries back to the original image coordinates
      global_xmin = np.min(white_pixel_x)
      global_ymin = np.min(white_pixel_y)
      global_xmax = np.max(white_pixel_x)
      global_ymax = np.max(white_pixel_y)

    # 7. Draw the bounding box on the original image canvas
      top_left_pt = (global_xmin, global_ymin)
      bottom_right_pt = (global_xmax, global_ymax)
    
      cv2.rectangle(img, top_left_pt, bottom_right_pt, (0,0,255), 1)
cv2.imwrite('output.jpg', img)


True

In [ ]:
a = 0
measurements = []
for i, cnt  in enumerate(contours):
  area = cv2.contourArea(cnt)

  if area > -100:
    rect = cv2.minAreaRect(cnt)
    box = cv2.boxPoints(rect)
    box = np.int64(box)
    cv2.drawContours(img, [box], 0, (0,0,255), 1)
    side1 = np.linalg.norm(box[0] - box[1])
    side2 = np.linalg.norm(box[1] - box[2])
    length = max(side1, side2)
    width = min(side1, side2)
    length = np.int64(length)
    width = np.int64(width)
    measurements.append([length, width, area])
    a = a + 1
    x, y, w, h = cv2.boundingRect(cnt)
    cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)
    cX = w// 2 + x
    cY = h// 2 + y
    cv2.putText(
      img, 
      text=f"#{a}", 
      org=(cX - 10, cY + 5), 
      fontFace=cv2.FONT_HERSHEY_SIMPLEX, 
      fontScale=0.5, 
      color=(255, 0, 0), 
      thickness=2
    )
    yn, ym = y - 2, x + h + 2
    xn, xm = x - 2, x + w + 2
    roi = image[yn:ym, xn:xm]
    _, bin = cv2.threshold(roi, 200, 255, cv2.THRESH_BINARY)
    roi_edge = cv2.Canny(bin, threshold1=50, threshold2=150)
    
    lines = cv2.HoughLinesP(roi_edge, 1, np.pi/180, threshold=200, minLineLength=200, maxLineGap=1)

    if lines is not None:
    
      pixel_row = np.sum(bin, axis = 1)
      y_pos = np.argmax(pixel_row)
      column = np.where(bin[y_pos, :] == 255)[0]
      xe = column[-1]
      xs = column[0]
      scalebar_length = xe - xs + 1
    #tnt, _ = cv2.findContours(bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    #if len(tnt) >= 2:
      #combined_pts = np.vstack((tnt[0], tnt[1]))
      #x, y, w, h = cv2.boundingRect(combined_pts)
      #cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)

True